# Oracle 26ai Banking Nudges — Self-Contained Training Notebook

This notebook is a single, runnable training artifact for the proactive banking nudges demo on Oracle Database 26ai. It replaces the need to open multiple README files, SQL scripts, and code folders, while still pointing to external assets only where they truly belong outside a notebook (raw dataset downloads, APEX export, and Spring/Java examples).
# What You Will Learn

By the end of this notebook you will understand how to:

- Use Oracle 26ai as one surface for relational, vector, graph, and AI workloads.
- Load and call an ONNX embedding model inside the database.
- Build a property graph overlay on existing relational tables with SQL/PGQ.
- Configure Select AI so natural language can generate SQL against your schema.
- Wire the database to an MCP-compatible agent and to an APEX chat UI.
- Implement three real-time nudge use cases.
- Operate the result with latency, capacity, security, and graceful-degradation guardrails.
# The Business Problem and Target Moments

The demo targets three high-intent banking moments:

1. **Credit card product page view** — assist while the customer's intent is fresh.
2. **Application abandonment** — recover the customer before intent decays.
3. **Declined transaction** — resolve the issue quickly with a concrete next action.

Success condition: decision latency must be low enough to act before the session or context is lost.
# Core Design Principle

**One operational surface.** Keep the source of truth where it already lives. Extend existing tables with vector columns and graph overlays rather than introducing separate vector and graph stores. Execute mixed retrieval in one query path with ACID guarantees. Only add external systems when scale or autonomy requirements are proven.
# The "AI" Is Just New Datatypes and Operators

From a DBA perspective, the AI pieces are concrete database objects:

- `VECTOR` column = fixed-length `FLOAT32` array.
- ONNX model = stored function loaded via `DBMS_VECTOR.LOAD_ONNX_MODEL`.
- Vector index = a new index type for approximate nearest-neighbor search.
- `VECTOR_EMBEDDING(... USING ... AS DATA)` and `VECTOR_DISTANCE(..., ..., COSINE)` = new SQL operators.
- Property Graph = a view-like overlay on relational tables.
- Select AI profile = a `DBMS_CLOUD_AI` package configuration.
- MCP server = a listener that exposes database tools to an LLM.

# Why This Matters: A Nudge Is a Regulated Communication

The demo looks like a recommendation engine, but in a bank every generated message is a regulated communication. Before writing any SQL, keep three mental models in mind:

1. **Oracle 26ai does not add a new database.** It adds a `VECTOR` datatype, a new index kind, a few SQL operators, a graph-view layer, and two PL/SQL packages (`DBMS_VECTOR`, `DBMS_CLOUD_AI`). Everything else — transactions, RBAC, partitioning, backup, audit, retention — works the way it already does.

2. **A nudge is a regulated communication, not a string.** It is the output of an end-to-end record: who was eligible, why, which data was retrieved, which model version generated the text, who reviewed it, who was suppressed, and on which channel it was delivered. If you cannot reproduce that record, you cannot ship.

3. **The regulatory map shapes the design.** The table below is why the notebook builds the schema, the vector index, the graph, the Select AI profile, and the MCP tools the way it does.

| Regime | Applies to | What it forces in the design |
|---|---|---|
| **UDAAP** | Any consumer-facing message, including LLM-generated nudges | Reproducible content record; no deceptive framing; human review queue |
| **Reg B / ECOA** | Credit decisions (Cash+ Visa, Personal Loan) | No protected-class features or proxies in eligibility; graph-derived audiences need justification |
| **FCRA** | Adverse action on credit | Specific reasons, not LLM rationale; no black-box denial |
| **Reg Z / Reg DD** | Credit-card / loan / deposit disclosures | APR/APY terms come from approved disclosures, not the LLM |
| **Reg E** | Declined transaction servicing | UC3 is servicing, not marketing — different consent and retention rules |
| **GLBA** | NPI in transcripts, account data, balances | Embeddings and prompts stay inside the ADB perimeter; encryption at rest + in transit |
| **TCPA / CAN-SPAM** | SMS, email, push | Channel consent, opt-out, frequency cap, quiet hours |
| **GDPR / CCPA** | EU/CA customers | Lawful basis, DSAR, right to deletion propagates to embeddings |
| **SR 11-7 / OCC 2011-12** | Embedding model + LLM as models | Model inventory, validation, monitoring, change control |
| **BSA / AML** | Transaction monitoring | Nudges must not leak SAR-related signals; fraud-decline messaging is constrained |
| **PCI-DSS** | PAN, CVV, expiry | Never embed PAN; tokenize before any AI surface |
| **NYDFS Part 500 / FFIEC** | Cybersecurity & third-party risk | LLM provider is a third party; data-flow inventory; incident reporting clock |
| **SOX** | Financial reporting | Campaign attribution feeding revenue recognition needs ITGC controls |
| **Records management** | Bank policy + regulator retention schedules | `AI_CALL_LOG`, prompts, outputs, eligibility snapshots are records; retention + legal hold |

Every technical choice in this notebook is made so that the final design can pass Architecture Review, Model Risk, Compliance, and Audit. Keep that lens on as you run each cell.

## Who this notebook is for

You are likely responsible for one or more of:

- **Offer eligibility & decisioning** — who is allowed to see the Cash+ Visa Intro APR offer right now?
- **Personalization & content generation** — what wording goes on the screen, in-app message, email, or push?
- **Suppression & opt-out enforcement** — who must *not* be marketed to today, and on which channels?
- **Channel-of-record** — is this a marketing communication (CAN-SPAM / TCPA / GLBA opt-out) or a transactional/servicing message (Reg E error resolution, Reg Z account servicing)?
- **Fair-lending & UDAAP review** — can you defend, in writing, why this customer got this offer, in this language, on this date?
- **Operations** — SLOs, on-call, model drift, audit trail, retention, legal hold, regulator data requests.

You are assumed to be comfortable with production Java/Spring Boot, operating Oracle, and the bank's existing offers stack. You do not need to be an expert in vectors, embeddings, ANN indexes, RAG, LLM generation, agents, or MCP — this notebook translates each one to something already in your operating model.

# Architecture

```mermaid
flowchart TD
    Datasets[Public Datasets: PaySim, LendingClub, Banking77, UCI] --> Scripts[scripts/01..04]
    Scripts --> Obj[OCI Object Storage]
    Obj --> Load[DBMS_CLOUD.COPY_DATA]
    Load --> Staging[STG_* Tables]
    Staging --> Transform[sql/05_transform.sql]
    Transform --> Core[(CUSTOMER ACCOUNT TXN APPLICATION PRODUCT OFFER PAGE_EVENT CONVERSATION)]
    Core --> Vector[CONVERSATION_CHUNK + VECTOR INDEX]
    Core --> Graph[banking_graph via SQL/PGQ]
    Core --> SelectAI[DBMS_CLOUD_AI profile NUDGE_BOT]
    APEX[APEX chat page] --> Core
    MCP[SQLcl MCP server] --> Core
```

# Environment Prerequisites

Before running cells you need:

- Python 3.10+ with `oracledb`, `pandas`, `numpy`, `python-dotenv`, and optionally `langchain-oracledb`.
- Oracle Database 26ai: local Docker/Podman container, Autonomous Database Free Tier, or ADB-S/dedicated.
- For ADB: either a wallet directory with `TNS_ADMIN` pointing to it, or a TLS-only `tnsnames.ora` directory if your ADB supports one-way TLS.
- For model download: `docker`/`podman` for local Oracle, or `DBMS_CLOUD.GET_OBJECT` for ADB.
- For Select AI: an OCI GenAI credential (`OCI_GENAI_CRED`) created in the database.
- For MCP: SQLcl 24+ installed locally.
- For dataset ingestion: Kaggle API key and optionally OCI CLI for object-storage uploads.
- For operations: OpenTelemetry collector or exporter endpoint, Micrometer-compatible metrics sink, and SIEM ingestion for unified audit.

Create a `.env` file in the notebook folder if you are not using Codespaces secrets:

```text
ORACLE_USER=testuser
ORACLE_PASSWORD=TestPass123
ORACLE_DSN=localhost:1521/FREEPDB1
ORACLE_MODEL_NAME=MINILM_EMB
ORACLE_ONNX_FILE=all_MiniLM_L6_v2.onnx
ORACLE_DIRECTORY_NAME=ONNX_DIR
BANKING_DEMO_ROOT=/workspaces/oracle-26ai-learning/26ai-banking-demo
TNS_ADMIN=/path/to/wallet
```

## Notebook roadmap

| Section | What you will learn |
|---|---|
| Schema + staging | How the AI surface extends existing relational tables |
| Embeddings + vector index | How `VECTOR`, `VECTOR_EMBEDDING`, and ANN indexes work |
| Property graph | How SQL/PGQ creates look-alike audiences without a graph DB |
| Select AI | How to govern LLM-generated nudges |
| Use cases 1–3 | How graph + vector + relational retrieval combine |
| APEX / MCP / Spring | How front-ends and agents connect |
| Capacity + operations | How to size, monitor, and audit the system |

# Configure GitHub Codespaces Secrets

When running this notebook in a GitHub Codespace, store credentials as Codespace secrets instead of committing them:

1. Go to your personal **GitHub settings → Codespaces → Secrets**.
2. Add a secret for each value below. They become environment variables when the Codespace starts:

| Secret | Environment variable | Example value |
|---|---|---|
| `ORACLE_USER` | `ORACLE_USER` | `ADMIN` or `TESTUSER` |
| `ORACLE_PASSWORD` | `ORACLE_PASSWORD` | your database password |
| `ORACLE_DSN` | `ORACLE_DSN` | `localhost:1521/FREEPDB1` or `nudgedb_high` |
| `TNS_ADMIN` | `TNS_ADMIN` | `/home/codespace/wallet` |

3. Rebuild or restart the Codespace so the secrets are exported as environment variables.
4. Do not place wallet ZIP files, TLS config directories (for example `wallet_tls/`), `.sso`, `.pem`, `.p12`, `.ora`, or `.env` files inside the repo. They will be blocked by `.gitignore`.

The next cell verifies the secrets are present and masks the password before falling back to `.env`.

# Alternative: Connect with TLS Instead of a Wallet

Oracle Autonomous Database can accept TLS (one-way TLS) connections without requiring a downloaded wallet, depending on your ADB network and security configuration. If TLS is enabled, you can connect using a TLS connection string and skip the `Wallet_*.zip` entirely.

## How to get the TLS connection string

1. In the OCI Console, open your Autonomous Database details page.
2. Click **Database connection**.
3. Under **Connection strings**, switch the **TLS authentication** option to **TLS** instead of **mTLS** if your tenancy/ADB supports it.
4. Copy the **TLS connection string**. It looks similar to:

```text
adbname_high =
  (description=
    (retry_count=20)(retry_delay=3)
    (address=(protocol=tcps)(port=1522)(host=adb.<region>.oraclecloud.com))
    (connect_data=(service_name=<service_name>))
    (security=(ssl_server_cert_dn="CN=adb.<region>.oraclecloud.com")
              (ssl_server_auto_dn_match=yes))
  )
```

## How to use it in this notebook

Create a directory (for example `/home/codespace/wallet_tls`) and place only a `tnsnames.ora` file inside it with the TLS descriptor above. Then set:

```text
TNS_ADMIN=/home/codespace/wallet_tls
ORACLE_DSN=adbname_high
```

`python-oracledb` thin mode uses the operating-system certificate store to validate the Oracle server certificate, so no `cwallet.sso`, `ewallet.p12`, or `sqlnet.ora` is required.

## Caveats

- TLS without a wallet is not available for every Autonomous Database deployment. If the console does not show a TLS option, continue with the wallet.
- Some organizations require mTLS with a wallet for compliance even when TLS is technically available.
- If you see `ORA-28759: failure to open file` or certificate-validation errors, the OS trust store may be missing the required CA. In that case either install the CA into the system store or fall back to the wallet.
- The `TNS_ADMIN` secret is still useful because it points to whichever directory holds your `tnsnames.ora` (wallet or TLS-only).

In [1]:
import os
from pathlib import Path

# Secrets priority: environment variables (GitHub Codespaces) -> .env file -> defaults.
required = ["ORACLE_USER", "ORACLE_PASSWORD", "ORACLE_DSN", "TNS_ADMIN"]
env_dotenv = Path(".env")
if env_dotenv.exists():
    for line in env_dotenv.read_text().splitlines():
        if line.strip() and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip())

missing = [s for s in required if not os.environ.get(s)]
if missing:
    print("WARNING: missing secrets/env vars:", missing)
    print("Set them via GitHub Codespaces Secrets or add a .env file.")
else:
    print("All required secrets are present.")
    print(f"ORACLE_USER  = {os.environ.get('ORACLE_USER')}")
    print(f"ORACLE_DSN   = {os.environ.get('ORACLE_DSN')}")
    print(f"TNS_ADMIN    = {os.environ.get('TNS_ADMIN')}")
    print(f"ORACLE_PASSWORD = {'*' * len(os.environ.get('ORACLE_PASSWORD', ''))}")

Set them via GitHub Codespaces Secrets or add a .env file.


In [2]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'oracledb', 'pandas', 'numpy', 'python-dotenv',
                'langchain', 'langchain-core', 'langchain-oracledb'],
               check=False)
print("Dependencies installed.")


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Dependencies installed.


In [ ]:
import os
import re
import oracledb
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

user = os.environ.get("ORACLE_USER", "testuser")
password = os.environ.get("ORACLE_PASSWORD", "TestPass123")
dsn = os.environ.get("ORACLE_DSN", "localhost:1521/FREEPDB1")
model_name = os.environ.get("ORACLE_MODEL_NAME", "MINILM_EMB")
onnx_file = os.environ.get("ORACLE_ONNX_FILE", "all_MiniLM_L6_v2.onnx")
directory_name = os.environ.get("ORACLE_DIRECTORY_NAME", "ONNX_DIR")
demo_root = os.environ.get("BANKING_DEMO_ROOT", "/workspaces/oracle-26ai-learning/26ai-banking-demo")
tns_admin = os.environ.get("TNS_ADMIN")

if tns_admin:
    os.environ["TNS_ADMIN"] = tns_admin

if user in ("testuser", "ADMIN") and password in ("TestPass123", "Welcome12345#"):
    print("WARNING: using default credentials. Set ORACLE_USER/ORACLE_PASSWORD via Codespaces secrets or .env.")

conn = oracledb.connect(user=user, password=password, dsn=dsn)

def run_sql(sql, params=None, fetch=True):
    with conn.cursor() as cur:
        params = params or {}
        # DDL scripts often contain multiple statements separated by semicolons.
        # Execute them one at a time when there are no bind parameters and the
        # text is not a PL/SQL block (which has its own internal semicolons).
        if not fetch and not params and ';' in sql.strip() and not re.search(r'\b(BEGIN|DECLARE)\b', sql, re.IGNORECASE):
            for stmt in [s.strip() for s in sql.split(';') if s.strip()]:
                cur.execute(stmt)
            return None
        cur.execute(sql, params)
        if fetch and cur.description:
            columns = [col[0] for col in cur.description]
            return pd.DataFrame(cur.fetchall(), columns=columns)
        return None

version = run_sql("SELECT * FROM v$version WHERE banner LIKE 'Oracle%'").iloc[0, 0]
print(f"Connected to: {dsn}")
print(f"Oracle version: {version}")

OperationalError: DPY-6005: cannot connect to database (CONNECTION_ID=I++Gd3+VgXjXK+8yrvl08Q==).
[Errno 111] Connection refused

# Download and Stage Public Datasets

Kaggle credentials and multi-gigabyte downloads belong outside the notebook. Run the repo scripts from the terminal:

```bash
./26ai-banking-demo/scripts/00_setup_kaggle.sh
./26ai-banking-demo/scripts/01_download_all.sh
python3 26ai-banking-demo/scripts/02_trim_lending.py \
  --input 26ai-banking-demo/data/raw/lendingclub/accepted_2007_to_2018Q4.csv \
  --output 26ai-banking-demo/data/processed/lendingclub_5k.csv
python3 26ai-banking-demo/scripts/03_gen_conversations.py \
  --input 26ai-banking-demo/data/raw/banking77/banking77.csv \
  --output 26ai-banking-demo/data/processed/banking77_conversations.csv
```

The next cell verifies the expected files exist.

In [ ]:
from pathlib import Path
import pandas as pd

demo_root = Path(os.environ.get("BANKING_DEMO_ROOT", "/workspaces/oracle-26ai-learning/26ai-banking-demo"))
expected = {
    "paysim": demo_root / "data/raw/paysim/PS_20174392719_1491204439457_log.csv",
    "lendingclub": demo_root / "data/processed/lendingclub_5k.csv",
    "banking77": demo_root / "data/processed/banking77_conversations.csv",
    "marketing": demo_root / "data/raw/marketing/bank-additional-full.csv",
}

for name, path in expected.items():
    if path.exists():
        rows = sum(1 for _ in open(path)) - 1
        print(f"{name}: OK ({rows:,} rows) -> {path}")
    else:
        print(f"{name}: MISSING -> {path}")

# Upload to OCI Object Storage (Optional)

If you are using Autonomous Database, upload the processed CSVs to an OCI bucket first:

```bash
OCI_NAMESPACE=<namespace> OCI_BUCKET_NAME=<bucket> ./26ai-banking-demo/scripts/04_upload_to_oci.sh
```

If you are running the local Docker Oracle image, you can skip OCI and load data via external tables or direct `pandas` inserts shown in the fallback path below.

# Data Model Evolution: Extend, Don't Replace

The schema you are about to create is intentionally boring. `CUSTOMER`, `ACCOUNT`, `TXN`, `APPLICATION`, `PRODUCT`, and `OFFER` are ordinary relational tables. The AI pieces are additions, not replacements:

| Existing asset | 26ai addition | Why |
|---|---|---|
| Customer / account / txn / application tables | No structural replacement | Preserve contracts with upstream systems |
| Conversation transcripts (`CONVERSATION.transcript`) | `CONVERSATION_CHUNK.embedding VECTOR(384, FLOAT32)` | Turn CLOB text into a typed, indexable semantic column |
| Product docs and offer content (`details_blob` / `details_text`) | `VECTOR` column (future) | Match offers to live browsing context |
| Foreign-key relationships across tables | `BANKING_GRAPH` property graph | Traverse relationships without ETL to a separate graph store |
| Static offer metadata | Select AI profile `NUDGE_BOT` | Generate offer wording from approved templates against live data |
| Stored procedures / SQL | MCP tools via SQLcl `-mcp` | Expose a fixed, least-privilege tool catalog to an LLM agent |
| OLTP change events | GoldenGate 23ai Distributed AI (future) | Real-time CDC + auto-embedding across regions |
| Large-scale filtered ANN | Exadata Smart Scan for Vectors (future) | Push vector distance + filters to storage cells |

The design principle is **one operational surface**. Upstream OLTP systems keep their contracts. The offers/personalization team extends the same schema with vector columns and graph overlays. Mixed retrieval runs in one SQL path with ACID guarantees.

## Converged vs. best-of-breed

| Concern | Best-of-breed stack | Oracle 26ai converged |
|---|---|---|
| Data movement | ETL to vector DB + graph DB | None — same row |
| Consistency | Eventual, app-managed | ACID across vector + relational |
| Security / PII | Multiple perimeters | One — VPD, redaction, audit |
| Latency | Network hops between stores | Single SQL, Smart Scan |
| Ops | 3–4 systems to run | 1 |
| Skills | New stacks | SQL + PL/SQL the team knows |

The bank's existing controls (encryption, key management, audit, retention, legal hold, change management, ITGC) already cover the AI surface — *if* you keep the AI surface inside the database. That is why this design is defensible to Compliance and Audit.

# Staging Pattern: Land Raw Data Before Normalizing

The staging tables below are a standard ELT pattern. They hold the raw public datasets in their original shape before the transform step maps them to the normalized core schema.

Why this matters for a production pipeline:

- **Data lineage:** a regulator can trace a row in `CUSTOMER` back to a row in `STG_PAYSIM`.
- **Reproducibility:** if the transform logic changes, you re-run from staging without re-downloading multi-gigabyte files.
- **Quality gates:** staging is where you validate schema, counts, and outliers before the data reaches the core tables.

The demo uses PaySim for transactions, LendingClub for applications, Banking77 for conversations, and UCI Bank Marketing for optional propensity features. Each source has its own license — review `26ai-banking-demo/docs/dataset-licenses.md` before using them outside this learning exercise.

In [ ]:
schema_sql = """
CREATE TABLE customer (
  customer_id    NUMBER PRIMARY KEY,
  full_name      VARCHAR2(120),
  segment        VARCHAR2(40),
  signup_date    DATE
);

CREATE TABLE product (
  product_id     NUMBER PRIMARY KEY,
  name           VARCHAR2(120),
  family         VARCHAR2(40),
  details_blob   BLOB,
  details_text   CLOB
);

CREATE TABLE offer (
  offer_id          NUMBER PRIMARY KEY,
  product_id        NUMBER REFERENCES product(product_id),
  offer_name        VARCHAR2(120),
  eligibility_rule  VARCHAR2(400),
  outcome_label     VARCHAR2(40)
);

CREATE TABLE account (
  account_id     NUMBER PRIMARY KEY,
  customer_id    NUMBER REFERENCES customer(customer_id),
  product_id     NUMBER REFERENCES product(product_id),
  daily_limit    NUMBER,
  opened_at      DATE
);

CREATE TABLE txn (
  txn_id          NUMBER PRIMARY KEY,
  account_id      NUMBER REFERENCES account(account_id),
  amount          NUMBER,
  status          VARCHAR2(20),
  decline_reason  VARCHAR2(80),
  txn_ts          TIMESTAMP
);

CREATE TABLE application (
  app_id         NUMBER PRIMARY KEY,
  customer_id    NUMBER REFERENCES customer(customer_id),
  product_id     NUMBER REFERENCES product(product_id),
  status         VARCHAR2(20),
  fields_json    JSON,
  updated_at     TIMESTAMP
);

CREATE TABLE page_event (
  event_id       NUMBER PRIMARY KEY,
  customer_id    NUMBER REFERENCES customer(customer_id),
  product_id     NUMBER REFERENCES product(product_id),
  page_url       VARCHAR2(400),
  event_ts       TIMESTAMP
);

CREATE TABLE conversation (
  conv_id        NUMBER PRIMARY KEY,
  customer_id    NUMBER REFERENCES customer(customer_id),
  channel        VARCHAR2(20),
  transcript     CLOB,
  conv_ts        TIMESTAMP
);

CREATE TABLE conversation_chunk (
  chunk_id       NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  conv_id        NUMBER REFERENCES conversation(conv_id),
  chunk_text     VARCHAR2(4000),
  embedding      VECTOR(384, FLOAT32)
);
"""

run_sql(schema_sql, fetch=False)
print("Core schema created.")
print("Tables: customer, product, offer, account, txn, application, page_event, conversation, conversation_chunk")

In [ ]:
staging_sql = """
CREATE TABLE stg_paysim (
  step               NUMBER,
  type               VARCHAR2(20),
  amount             NUMBER,
  name_orig          VARCHAR2(40),
  oldbalance_org     NUMBER,
  newbalance_orig    NUMBER,
  name_dest          VARCHAR2(40),
  oldbalance_dest    NUMBER,
  newbalance_dest    NUMBER,
  is_fraud           NUMBER,
  is_flagged_fraud   NUMBER
);

CREATE TABLE stg_lending (
  id               NUMBER,
  member_id        NUMBER,
  loan_amnt        NUMBER,
  term             VARCHAR2(30),
  int_rate         VARCHAR2(20),
  grade            VARCHAR2(5),
  sub_grade        VARCHAR2(5),
  emp_length       VARCHAR2(30),
  home_ownership   VARCHAR2(30),
  annual_inc       NUMBER,
  purpose          VARCHAR2(100),
  loan_status      VARCHAR2(80),
  issue_d          VARCHAR2(20)
);

CREATE TABLE stg_banking77 (
  text             VARCHAR2(500),
  category         VARCHAR2(60)
);

CREATE TABLE stg_marketing (
  age               NUMBER,
  job               VARCHAR2(40),
  marital           VARCHAR2(20),
  education         VARCHAR2(40),
  "default"         VARCHAR2(10),
  housing           VARCHAR2(10),
  loan              VARCHAR2(10),
  contact           VARCHAR2(20),
  month             VARCHAR2(10),
  day_of_week       VARCHAR2(10),
  duration          NUMBER,
  campaign          NUMBER,
  pdays             NUMBER,
  previous          NUMBER,
  poutcome          VARCHAR2(20),
  emp_var_rate      NUMBER,
  cons_price_idx    NUMBER,
  cons_conf_idx     NUMBER,
  euribor3m         NUMBER,
  nr_employed       NUMBER,
  y                 VARCHAR2(10)
);
"""

run_sql(staging_sql, fetch=False)
print("Staging tables created: stg_paysim, stg_lending, stg_banking77, stg_marketing")

In [ ]:
try:
    # Path B: ADB / cloud — pull the model from Oracle's public bucket and load it.
    load_cloud_sql = """
    BEGIN
      DBMS_CLOUD.GET_OBJECT(
        credential_name => NULL,
        object_uri => 'https://objectstorage.us-phoenix-1.oraclecloud.com/n/adwc4pm/b/OML-Resources/o/all_MiniLM_L6_v2.onnx',
        directory_name => 'DATA_PUMP_DIR'
      );
    END;
    """
    run_sql(load_cloud_sql, fetch=False)

    run_sql("""
    BEGIN
      DBMS_VECTOR.LOAD_ONNX_MODEL(
        'DATA_PUMP_DIR',
        'all_MiniLM_L6_v2.onnx',
        'MINILM_EMB',
        JSON('{"function":"embedding","embeddingOutput":"embedding","input":{"input":["DATA"]}}')
      );
    END;
    """, fetch=False)
    print("ONNX model loaded via DBMS_CLOUD.GET_OBJECT.")
except Exception as e:
    print(f"Cloud model load failed (expected if using local Docker or missing privileges): {e}")
    print("For local Docker, download all_MiniLM_L6_v2.onnx, copy it into the container, and load from a custom DIRECTORY.")

models = run_sql("SELECT model_name, mining_function FROM user_mining_models WHERE model_name = 'MINILM_EMB'")
print(models)

# Embeddings: What They Are and Why Model Risk Matters

An **embedding** is a fixed-length float vector that represents the meaning of a piece of text. Similar text produces vectors that are close together in the vector space. For example, "my card was declined at the gas station" and "transaction failed at the pump" produce vectors that are very close, while "apply for a mortgage" is far away.

The ONNX model loaded below (`MINILM_EMB`) is the function that turns text into that vector. It runs in-process inside the database — no callout, no network — so embedding 20,000 rows is basically `INSERT ... SELECT VECTOR_EMBEDDING(...) FROM ...`.

## Mental translation for your existing stack

| AI concept | Equivalent in an offers/campaign stack |
|---|---|
| Embedding model | A deterministic feature extractor, like a look-alike hashing function, but emitting 384 floats instead of a hash |
| `VECTOR(384, FLOAT32)` column | A strongly typed feature column. It is **derived NPI** — same controls as the source `transcript` |
| ANN vector index | A new access path, like a B-tree, but for nearest-neighbor instead of equality/range |
| Cosine distance | "How close is this customer's language to that snippet?" Used for candidate generation, not the final yes/no |

## Model-risk checklist

Because the embedding model scores customer language, it is a model in use under SR 11-7 / OCC 2011-12:

- Put `MINILM_EMB` in the model inventory with owner, version, validation report, intended use, and documented limitations. "It's just open-source MiniLM" is not a defense.
- Hash and sign the ONNX file as a deployment artifact; version it next to your DDL.
- Loading the model in-database keeps transcript bytes inside the ADB perimeter. Document this in the data-flow inventory you maintain for NYDFS 500 / third-party risk.
- The 3500-character truncation in the embed step is a silent control — anything beyond that is dropped. Document it in the data dictionary. A UDAAP risk exists if a customer's later sentences (e.g., "I told you I couldn't afford it") sit beyond the cutoff and are ignored.
- `CONVERSATION_CHUNK` duplicates NPI from `CONVERSATION`. It must inherit the same TDE encryption, retention, legal hold, and right-to-erasure propagation.
- No `WHERE` filter on the embed step means every conversation is embedded, including customers who later opt out. Either gate the insert on `customer.personalization_opt_in = 'Y'` or enforce suppression at retrieval time (Module 4).

## Operational checklist for Module 1

- [ ] `MINILM_EMB` is in the model inventory.
- [ ] ONNX file is hashed/signed and stored as a release artifact.
- [ ] Embedding step runs inside the ADB perimeter (no transcript bytes leave).
- [ ] `CONVERSATION_CHUNK` is in the same TDE-encrypted tablespace as `CONVERSATION`.
- [ ] Retention policy on `CONVERSATION_CHUNK` matches `CONVERSATION`.
- [ ] Erasure path deletes both `CONVERSATION` and `CONVERSATION_CHUNK` rows.
- [ ] Query metric matches index metric (cosine ↔ cosine).
- [ ] Canary recall@K measured per release.
- [ ] p50/p95/p99 latency baselined.
- [ ] Top-K query always JOINs to `customer` for opt-in / suppression gates.

# Vector Indexing: IVF vs. HNSW and the Hybrid-Retrieval Rule

The demo uses `ORGANIZATION NEIGHBOR PARTITIONS` (IVF-style) with `DISTANCE COSINE` and `TARGET ACCURACY 90`. That is the right default for this scale.

| Dimension | IVF (demo choice) | HNSW (`INMEMORY NEIGHBOR GRAPH`) |
|---|---|---|
| Memory footprint | Lower | Higher (in-memory graph) |
| Best for | Filtered top-K (e.g., "for this customer/segment, top-5 chunks") | Tight-latency, larger top-K, less filtering |
| UC1/UC2 fit | Good default — always filtered by `customer_id` or `product_id` | Often overkill |
| UC3 fit | Good with right partition count | Better if recall@1 matters most |

## The only pattern you should ship: hybrid retrieval

Never run a pure vector query against the whole corpus. Always narrow with relational predicates first. This is both a performance rule and a fair-lending rule: you do not want vector similarity to silently surface protected-class-correlated language as "similar."

A production query shape looks like:

```sql
SELECT cc.chunk_id, cc.chunk_text,
       VECTOR_DISTANCE(cc.embedding, VECTOR_EMBEDDING(MINILM_EMB USING :query_text AS DATA), COSINE) AS distance
FROM   conversation_chunk cc
JOIN   conversation c ON c.conv_id = cc.conv_id
JOIN   customer cu     ON cu.customer_id = c.customer_id
WHERE  cu.segment = :segment
  AND  cu.personalization_opt_in = 'Y'
  AND  NOT EXISTS (
    SELECT 1 FROM offer_suppression s
    WHERE  s.customer_id = cu.customer_id
  )
ORDER BY distance
FETCH FIRST :k ROWS ONLY;
```

The demo keeps the queries simple so they run end-to-end, but a real bank adds the opt-in and suppression gates. Module 4 (MCP) shows how to enforce them at the tool layer.

# Ingest Staging Data

For ADB, fill in the placeholders in `04_copy_data.sql` and run the cloud copy. The notebook defaults to a local pandas fallback that inserts into staging tables directly, which works for local Docker Oracle without object storage.

In [ ]:
import pandas as pd

def load_csv_to_oracle(path, table, columns=None, chunksize=1000, sep=","):
    if not path.exists():
        print(f"SKIP: {path} not found")
        return 0
    df = pd.read_csv(path, sep=sep, low_memory=False)
    if columns:
        df = df[columns]
    df = df.where(pd.notnull(df), None)
    rows = []
    with conn.cursor() as cur:
        for r in df.itertuples(index=False, name=None):
            rows.append(r)
            if len(rows) >= chunksize:
                cur.executemany(f"INSERT INTO {table} VALUES ({','.join([':' + str(i+1) for i in range(len(rows[0]))])})", rows)
                rows = []
        if rows:
            cur.executemany(f"INSERT INTO {table} VALUES ({','.join([':' + str(i+1) for i in range(len(rows[0]))])})", rows)
    conn.commit()
    print(f"Loaded {len(df)} rows into {table}")
    return len(df)

demo_root = Path(os.environ.get("BANKING_DEMO_ROOT", "/workspaces/oracle-26ai-learning/26ai-banking-demo"))

load_csv_to_oracle(demo_root / "data/raw/paysim/PS_20174392719_1491204439457_log.csv", "STG_PAYSIM",
                   columns=["step","type","amount","nameOrig","oldbalanceOrg","newbalanceOrig","nameDest","oldbalanceDest","newbalanceDest","isFraud","isFlaggedFraud"])
load_csv_to_oracle(demo_root / "data/processed/lendingclub_5k.csv", "STG_LENDING")
load_csv_to_oracle(demo_root / "data/processed/banking77_conversations.csv", "STG_BANKING77")
load_csv_to_oracle(demo_root / "data/raw/marketing/bank-additional-full.csv", "STG_MARKETING", sep=";")

print("Staging load complete.")

In [ ]:
transform_sql = """
INSERT INTO product (product_id, name, family, details_blob, details_text)
SELECT 1, 'Cash+ Visa', 'CREDIT_CARD', TO_BLOB(UTL_RAW.CAST_TO_RAW('Cash+ Visa product sheet')), TO_CLOB('Cash+ Visa with rewards and configurable categories') FROM dual
UNION ALL
SELECT 2, 'Personal Loan', 'LOAN', TO_BLOB(UTL_RAW.CAST_TO_RAW('Personal Loan brochure')), TO_CLOB('Personal Loan fixed term repayment product') FROM dual
UNION ALL
SELECT 3, 'Term Deposit', 'DEPOSIT', TO_BLOB(UTL_RAW.CAST_TO_RAW('Term Deposit fact sheet')), TO_CLOB('Term Deposit with fixed duration and fixed interest') FROM dual;

INSERT INTO offer (offer_id, product_id, offer_name, eligibility_rule, outcome_label)
SELECT 1, 1, 'Cash+ Visa Intro APR', 'segment in (Prime, Affluent)', 'N/A' FROM dual
UNION ALL
SELECT 2, 2, 'Personal Loan Cashback', 'application purpose in debt_consolidation', 'N/A' FROM dual
UNION ALL
SELECT 3, 3, 'Term Deposit Bonus Rate', 'new_to_bank = Y', 'N/A' FROM dual;

INSERT INTO customer (customer_id, full_name, segment, signup_date)
SELECT rn, name_orig,
       CASE MOD(rn, 3) WHEN 0 THEN 'Mass' WHEN 1 THEN 'Prime' ELSE 'Affluent' END,
       TRUNC(SYSDATE) - MOD(rn, 720)
FROM (
  SELECT name_orig, ROW_NUMBER() OVER (ORDER BY name_orig) rn
  FROM (SELECT DISTINCT name_orig FROM stg_paysim)
)
WHERE rn <= 500;

INSERT INTO account (account_id, customer_id, product_id, daily_limit, opened_at)
SELECT seed.lvl, seed.customer_id, seed.product_id,
       CASE c.segment WHEN 'Mass' THEN 2000 WHEN 'Prime' THEN 5000 WHEN 'Affluent' THEN 10000 ELSE 2000 END,
       TRUNC(SYSDATE) - MOD(seed.lvl, 900)
FROM (
  SELECT LEVEL lvl, MOD(LEVEL - 1, 500) + 1 AS customer_id, MOD(LEVEL - 1, 3) + 1 AS product_id
  FROM dual CONNECT BY LEVEL <= 800
) seed
JOIN customer c ON c.customer_id = seed.customer_id;

INSERT INTO txn (txn_id, account_id, amount, status, decline_reason, txn_ts)
SELECT rn, MOD(rn - 1, 800) + 1, amount,
       CASE WHEN NVL(is_fraud, 0) = 1 OR NVL(is_flagged_fraud, 0) = 1 THEN 'DECLINED' ELSE 'APPROVED' END,
       CASE WHEN NVL(is_flagged_fraud, 0) = 1 THEN 'LIMIT_EXCEEDED'
            WHEN NVL(is_fraud, 0) = 1 THEN 'SUSPECTED_FRAUD'
            ELSE NULL END,
       SYSTIMESTAMP - NUMTODSINTERVAL(MOD(step, 10080), 'MINUTE')
FROM (
  SELECT ROW_NUMBER() OVER (ORDER BY step, name_orig, name_dest) rn,
         step, amount, is_fraud, is_flagged_fraud
  FROM stg_paysim
)
WHERE rn <= 10000;

INSERT INTO application (app_id, customer_id, product_id, status, fields_json, updated_at)
SELECT rn, MOD(rn - 1, 500) + 1,
       CASE WHEN LOWER(NVL(purpose, '')) LIKE '%credit%' THEN 1 ELSE 2 END,
       CASE WHEN loan_status IN ('Current', 'Fully Paid') THEN 'SUBMITTED'
            WHEN loan_status = 'In Grace Period' THEN 'STARTED'
            ELSE 'ABANDONED' END,
       JSON_OBJECT('loan_amnt' VALUE loan_amnt, 'term' VALUE term, 'int_rate' VALUE int_rate,
                   'grade' VALUE grade, 'sub_grade' VALUE sub_grade, 'emp_length' VALUE emp_length,
                   'home_ownership' VALUE home_ownership, 'annual_inc' VALUE annual_inc,
                   'purpose' VALUE purpose, 'loan_status' VALUE loan_status, 'issue_d' VALUE issue_d),
       SYSTIMESTAMP - NUMTODSINTERVAL(MOD(rn, 2880), 'MINUTE')
FROM (
  SELECT ROW_NUMBER() OVER (ORDER BY id) rn, loan_amnt, term, int_rate, grade, sub_grade,
         emp_length, home_ownership, annual_inc, purpose, loan_status, issue_d
  FROM stg_lending
)
WHERE rn <= 5000;

INSERT INTO conversation (conv_id, customer_id, channel, transcript, conv_ts)
SELECT rn, MOD(rn - 1, 500) + 1, 'CHAT',
       TO_CLOB('Customer: ' || text || CHR(10) || 'Agent: [resolution for ' || category || ']'),
       SYSTIMESTAMP - NUMTODSINTERVAL(MOD(rn, 43200), 'MINUTE')
FROM (
  SELECT ROW_NUMBER() OVER (ORDER BY text) rn, text, category
  FROM stg_banking77
)
WHERE rn <= 10000;

INSERT INTO page_event (event_id, customer_id, product_id, page_url, event_ts)
SELECT lvl, TRUNC(DBMS_RANDOM.VALUE(1, 501)), TRUNC(DBMS_RANDOM.VALUE(1, 4)),
       CASE TRUNC(DBMS_RANDOM.VALUE(1, 4))
         WHEN 1 THEN '/products/cash-plus-visa'
         WHEN 2 THEN '/products/personal-loan'
         ELSE '/products/term-deposit' END,
       SYSTIMESTAMP - NUMTODSINTERVAL(TRUNC(DBMS_RANDOM.VALUE(1, 43200)), 'MINUTE')
FROM (SELECT LEVEL lvl FROM dual CONNECT BY LEVEL <= 1000);

COMMIT;
"""

run_sql(transform_sql, fetch=False)
counts = run_sql("""
SELECT 'customer' tbl, COUNT(*) cnt FROM customer
UNION ALL SELECT 'account', COUNT(*) FROM account
UNION ALL SELECT 'txn', COUNT(*) FROM txn
UNION ALL SELECT 'application', COUNT(*) FROM application
UNION ALL SELECT 'conversation', COUNT(*) FROM conversation
UNION ALL SELECT 'page_event', COUNT(*) FROM page_event
UNION ALL SELECT 'product', COUNT(*) FROM product
UNION ALL SELECT 'offer', COUNT(*) FROM offer
""")
print(counts)

In [ ]:
embed_sql = """
INSERT INTO conversation_chunk (conv_id, chunk_text, embedding)
SELECT c.conv_id,
       SUBSTR(c.transcript, 1, 3500),
       VECTOR_EMBEDDING(MINILM_EMB USING SUBSTR(c.transcript, 1, 3500) AS DATA)
FROM conversation c;

CREATE VECTOR INDEX conv_chunk_idx
ON conversation_chunk(embedding)
ORGANIZATION NEIGHBOR PARTITIONS
DISTANCE COSINE
WITH TARGET ACCURACY 90;

COMMIT;
"""

run_sql(embed_sql, fetch=False)
print("Embeddings and vector index created.")
print(run_sql("SELECT COUNT(*) chunks FROM conversation_chunk"))
print(run_sql("SELECT index_name, index_type FROM user_indexes WHERE index_name = 'CONV_CHUNK_IDX'"))

# Property Graph: Look-Alike Audiences Without a Separate Graph Database

SQL/PGQ creates a graph **view** over existing relational tables. No data is duplicated, no new backup strategy is needed, and no NPI is copied to another platform. The graph powers UC1: "Customers who viewed this card also viewed that card."

## Why this matters for offers

- **Reviewability:** Compliance can read the `MATCH` clause and see the peer logic in one line. Three nested self-joins are easy to misread.
- **Optimizer awareness:** the engine knows it is a traversal and can pick a better plan than the equivalent join chain.
- **Stability:** changing from 2-hop to 3-hop peer expansion is a one-token edit, not a refactor — easier to A/B test.
- **No new operational tier:** backup, replication, RAC, Data Guard, and AWR work exactly as they do for the underlying tables.

## What a principal engineer flags in the demo graph

1. **`ACCOUNT` is both a vertex table and an edge table.** That's deliberate, but it means the graph has two ways of asking "does this customer hold this product" and they must agree. Add a reconciliation check.
2. **`customer.segment` is exposed as a vertex property.** Segment is a business attribute, not a protected class — but if your bank's segment definition uses ZIP code, age band, or income proxies, segment becomes a protected-class proxy. Confirm with Compliance.
3. **`full_name` as a vertex property** is unnecessary for any graph query in the demo. The principle is data minimization on the graph surface.
4. **No edge from `customer` to `customer`.** Good. A direct customer-to-customer edge would invite collaborative-filtering on PII.

## Fair-lending warning

A graph traversal that influences a credit-product offer is a feature in a credit decision. That brings ECOA / Reg B into scope:

- Do not use protected-class attributes — or proxies like ZIP code, surname, language, or certain income bands — as graph properties or `MATCH` filters.
- The demo's UC1 peer traversal is symmetric on `viewed` only, with no reference to `segment`. Good. The moment someone adds `WHERE c1.segment = c2.segment`, you have segment-restricted peering and must justify it to Compliance for credit products.
- Graph should determine *which* offer to *show*, never *whether* to *approve*. Approval logic stays in deterministic, reviewable rules.
- Adverse-action defensibility: if a graph traversal contributes to a denial or non-presentation of a credit offer, you must state, in plain English, the specific reason at the level FCRA adverse-action notices require. "The graph said no" is not a reason.
- Segregate marketing vs. credit-decisioning paths into two named wrapper procedures with different `WHERE` constraints and different audit tags.

## Auditability — what to log per graph query

For every graph traversal that contributes to a customer-facing decision, log:

- `customer_id` (the subject of the decision),
- the MATCH pattern identifier (a stable name, not the raw SQL),
- input parameters,
- candidate set returned (IDs only, not PII),
- wall-clock time and trace ID,
- downstream decision (which offer, suppressed, control group, etc.).

This is the same pattern used for `AI_CALL_LOG`. Module 5 unifies them under one `OFFER_DECISION_LOG`.

## Performance guidance

- Index `SOURCE KEY` / `DESTINATION KEY` columns: `page_event(customer_id)`, `page_event(product_id)`, `application(customer_id)`, `application(product_id)`.
- Keep relational predicates selective and ahead of broad traversals. In UC1 the `c1.customer_id = :cid` predicate must be applied first — verify in `EXPLAIN PLAN`.
- Capture representative graph query plans in SQL Plan Management to flag regressions (Module 6).
- Multi-hop traversals (>2 hops) explode quickly. Cap with `FETCH FIRST` and add a max-hop guard in the wrapper package.

## Module 2 verify-yourself

- `BANKING_GRAPH` exists (`SELECT * FROM user_property_graphs`).
- `ACCOUNT` appears in both vertex and edge definitions.
- The UC1 MATCH pattern does not reference any protected-class attribute or known proxy.
- Indexes exist on every FK column used as `SOURCE KEY` / `DESTINATION KEY`.
- A credit-decision path cannot call graph traversals without a specific FCRA-grade reason code.

In [ ]:
graph_sql = """
CREATE PROPERTY GRAPH banking_graph
  VERTEX TABLES (
    customer KEY (customer_id) LABEL customer PROPERTIES (full_name, segment),
    product  KEY (product_id)  LABEL product  PROPERTIES (name, family),
    account  KEY (account_id)  LABEL account  PROPERTIES (daily_limit)
  )
  EDGE TABLES (
    account
      SOURCE KEY (customer_id) REFERENCES customer
      DESTINATION KEY (product_id) REFERENCES product
      LABEL holds,
    page_event
      KEY (event_id)
      SOURCE KEY (customer_id) REFERENCES customer
      DESTINATION KEY (product_id) REFERENCES product
      LABEL viewed PROPERTIES (event_ts),
    application
      KEY (app_id)
      SOURCE KEY (customer_id) REFERENCES customer
      DESTINATION KEY (product_id) REFERENCES product
      LABEL applied_for PROPERTIES (status)
  );
"""

run_sql(graph_sql, fetch=False)
print("Property graph created.")

sanity = run_sql("""
SELECT *
FROM GRAPH_TABLE(
  banking_graph
  MATCH (c IS customer)-[:viewed]->(p IS product)
  COLUMNS (c.full_name AS customer, p.name AS product)
)
FETCH FIRST 5 ROWS ONLY
""")
print(sanity)

In [ ]:
try:
    profile_sql = """
    BEGIN
      DBMS_CLOUD_AI.CREATE_PROFILE(
        profile_name => 'NUDGE_BOT',
        attributes   => '{
          "provider":"oci",
          "credential_name":"OCI_GENAI_CRED",
          "model":"cohere.command-r-plus",
          "object_list":[
            {"owner":"ADMIN","name":"CUSTOMER"},
            {"owner":"ADMIN","name":"TXN"},
            {"owner":"ADMIN","name":"APPLICATION"},
            {"owner":"ADMIN","name":"CONVERSATION_CHUNK"}
          ]
        }'
      );
    END;
    """
    run_sql(profile_sql, fetch=False)
    run_sql("BEGIN DBMS_CLOUD_AI.SET_PROFILE('NUDGE_BOT'); END;", fetch=False)
    print("Select AI profile NUDGE_BOT created and activated.")
except Exception as e:
    print(f"Select AI not configured (expected if OCI_GENAI_CRED is missing): {e}")
    print("Skip this step if you are not using OCI GenAI.")

# Select AI Profile Walkthrough

The profile below configures the `NUDGE_BOT` Select AI identity. It tells the database which LLM provider to use, which credential to use, which model to call, and which tables the LLM is allowed to see for natural-language grounding.

```sql
DBMS_CLOUD_AI.CREATE_PROFILE(
  profile_name => 'NUDGE_BOT',
  attributes   => '{
    "provider":"oci",
    "credential_name":"OCI_GENAI_CRED",
    "model":"cohere.command-r-plus",
    "object_list":[
      {"owner":"ADMIN","name":"CUSTOMER"},
      {"owner":"ADMIN","name":"TXN"},
      {"owner":"ADMIN","name":"APPLICATION"},
      {"owner":"ADMIN","name":"CONVERSATION_CHUNK"}
    ]
  }'
);
```

## What a principal engineer flags

1. **`object_list` is a data-minimization control.** It is the allow-list of tables the profile can see for NL→SQL grounding. Treat it like a privacy data map: every table on this list must have a documented reason. Remove `CUSTOMER.full_name` from the surface; use a view that exposes only `customer_id` and `segment`.
2. **`ADMIN` ownership in the demo is a Free-Tier convenience.** In a real bank, the profile's grants resolve through a least-privilege role, never `ADMIN`.
3. **The model name (`cohere.command-r-plus`) is a model-inventory entry.** It needs owner, version, vendor, intended use, region (data residency!), and tested limitations. SR 11-7 says so.
4. **The provider call leaves the bank.** Even on OCI GenAI, prompt + grounded data egress to the model service. Confirm the contractual data-handling terms cover NPI and the region is approved by the privacy office.
5. **No content/safety configuration in the profile alone.** The profile does not give you UDAAP-grade content filtering. That comes from your wrapper package and disclosure-substitution step.

## The wrapper package (`PKG_NUDGE_AI`)

Direct calls to `DBMS_CLOUD_AI.GENERATE` from application SQL are a control failure. Every call must go through a wrapper that:

- Re-runs opt-in / suppression / frequency-cap checks (defense in depth).
- Enforces the channel-of-record split (servicing vs. marketing).
- Looks up approved disclosure language for the offer's product (Reg Z APR, Reg DD APY) and **substitutes** it into the LLM output.
- Inserts an `AI_CALL_LOG` row before and after the call, with W3C trace context.
- Routes output to the UDAAP review queue when required (new template, new offer, new model, sampled).
- Catches errors and returns a safe deterministic fallback string.

## `AI_CALL_LOG` — records of record

This table lets you answer a regulator data request of the form "show me everything you ever sent to customer 1001":

```sql
CREATE TABLE ai_call_log (
  call_id            NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  created_at         TIMESTAMP DEFAULT SYSTIMESTAMP NOT NULL,
  customer_id        NUMBER,
  use_case           VARCHAR2(30),
  offer_id           NUMBER,
  channel            VARCHAR2(20),
  channel_of_record  VARCHAR2(20),
  profile_name       VARCHAR2(128),
  model_name         VARCHAR2(256),
  model_version      VARCHAR2(64),
  trace_id           VARCHAR2(64),
  span_id            VARCHAR2(32),
  prompt_template_id VARCHAR2(64),
  prompt_hash        VARCHAR2(128),
  prompt_tokens      NUMBER,
  output_tokens      NUMBER,
  output_hash        VARCHAR2(128),
  output_text        CLOB,
  disclosure_id      VARCHAR2(64),
  suppression_check  VARCHAR2(20),
  optin_check        VARCHAR2(20),
  freq_cap_check     VARCHAR2(20),
  control_group      VARCHAR2(20),
  review_queue_id    NUMBER,
  status             VARCHAR2(20),
  error_text         VARCHAR2(4000),
  retention_until    DATE
);
```

`output_text` is intentionally a CLOB, not just a hash — regulators have asked for the literal text. `retention_until` is computed from records-management policy at insert time. All three `*_check` columns must be `PASS` for `status = OK`.

## UDAAP review queue and disclosure substitution

A percentage of generated nudges (and 100% of new templates/offers/models) is routed to a human review queue. The LLM never paraphrases APR, APY, fees, or rate terms. The wrapper renders an approved template with a `{{disclosure_block}}` placeholder, calls `GENERATE`, then replaces the placeholder with pre-approved text from an `APPROVED_DISCLOSURES` table keyed by `offer_id` and effective date. If the placeholder is missing or a `%` token appears outside it, the wrapper rejects the output.

## Channel-of-record split

| UC | Channel of record | Most-relevant rules |
|---|---|---|
| UC1 — credit-card page view | Marketing | UDAAP, Reg Z if APR mentioned, TCPA/CAN-SPAM if pushed off-page |
| UC2 — abandoned application | Marketing | UDAAP, Reg Z, ECOA, CAN-SPAM/TCPA on email/SMS |
| UC3 — declined transaction | **Servicing (Reg E)** | Reg E error/dispute, BSA/AML (no SAR-leakage), GLBA. Marketing opt-out does **not** block it. |

# The End-to-End Offer Lifecycle

Every nudge in this stack follows the same lifecycle. The three use cases differ only in trigger and channel-of-record.

```text
 1. TRIGGER          page_event / abandoned application / declined txn
 2. RELATIONAL SCOPE customer state, account, segment, last interaction
 3. GRAPH CONTEXT    SQL/PGQ peer / look-alike candidates (UC1)
 4. VECTOR RETRIEVAL similar past conversation snippets (all UCs)
 5. ELIGIBILITY      deterministic OFFER.eligibility_rule check
 6. SUPPRESSION      opt-in + do-not-contact + offer-suppression list
 7. FREQUENCY CAP    rolling-window send count by channel
 8. CHANNEL OF RECORD  marketing vs. servicing routing
 9. CONTROL GROUP    holdout assignment (causal attribution)
10. GENERATION       PKG_NUDGE_AI.GENERATE with approved template
11. DISCLOSURE SUB.  Reg Z / Reg DD approved language injection
12. UDAAP REVIEW     route to queue if sampled / new template / new model
13. DELIVERY         channel-specific dispatch
14. ATTRIBUTION      response/conversion captured
15. ARCHIVAL         AI_CALL_LOG retention + legal-hold awareness
```

Steps 1–4 are candidate generation. Steps 5–7 are gating. Step 8 determines the rulebook. Steps 14–15 are the record. Skipping any step is a compliance finding.

## Use-case specifics

| UC | Trigger | Channel of record | Key control |
|---|---|---|---|
| UC1 — card page view | `/products/cash-plus-visa` | Marketing | Reg Z APR language from approved disclosures |
| UC2 — abandoned application | `status='STARTED'` older than 1 hour | Marketing | ECOA: eligibility rule on Personal Loan must be reviewable |
| UC3 — declined transaction | `txn.status='DECLINED'` | **Servicing (Reg E)** | BSA/AML: do not reveal SAR-relevant fraud detail |

## The `OFFER_DECISION_LOG`

Module 5 unifies every decision into one table:

```sql
CREATE TABLE offer_decision_log (
  decision_id        NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  decided_at         TIMESTAMP DEFAULT SYSTIMESTAMP NOT NULL,
  customer_id        NUMBER,
  use_case           VARCHAR2(30),
  trigger_event_id   NUMBER,
  candidate_offers   VARCHAR2(400),
  chosen_offer_id    NUMBER,
  decision           VARCHAR2(30),  -- ELIGIBLE / NOT_ELIGIBLE / SUPPRESSED / FREQ_CAPPED / HOLDOUT / SENT / FALLBACK / ERROR
  decision_reason    VARCHAR2(400),
  channel            VARCHAR2(20),
  channel_of_record  VARCHAR2(20),
  control_group      VARCHAR2(20),
  ai_call_id         NUMBER,
  trace_id           VARCHAR2(64),
  retention_until    DATE
);
```

`decision_reason` must be populated even on negative outcomes (`NOT_ELIGIBLE`, `SUPPRESSED`, `FREQ_CAPPED`). A regulator will ask "why didn't customer 1001 see the offer?" as often as "why did they?".

## Control groups and attribution

Every offer needs a holdout. Without a holdout you cannot prove the offer caused the outcome, and attribution claims may overstate value (a SOX issue if revenue recognition relies on them).

- Holdout assignment is deterministic from `(customer_id, offer_id)`, e.g., `MOD(ORA_HASH(customer_id || ':' || offer_id), 100) < holdout_pct`.
- Holdout customers go through every step except generation+delivery. Their log row has `control_group = 'HOLDOUT'`, `decision = 'HOLDOUT'`, `ai_call_id IS NULL`.
- Attribution joins `OFFER_DECISION_LOG` to downstream outcome events within an attribution window.

The code cells below run each use case. The simple versions skip suppression and disclosure substitution so they fit in a learning notebook, but the sections above show what production hardening looks like.

In [ ]:
governance_statements = [
    """CREATE TABLE ai_call_log (
        call_id            NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        created_at         TIMESTAMP DEFAULT SYSTIMESTAMP NOT NULL,
        customer_id        NUMBER,
        use_case           VARCHAR2(30),
        offer_id           NUMBER,
        channel            VARCHAR2(20),
        channel_of_record  VARCHAR2(20),
        profile_name       VARCHAR2(128),
        model_name         VARCHAR2(256),
        model_version      VARCHAR2(64),
        trace_id           VARCHAR2(64),
        span_id            VARCHAR2(32),
        prompt_template_id VARCHAR2(64),
        prompt_hash        VARCHAR2(128),
        prompt_tokens      NUMBER,
        output_tokens      NUMBER,
        output_hash        VARCHAR2(128),
        output_text        CLOB,
        disclosure_id      VARCHAR2(64),
        suppression_check  VARCHAR2(20),
        optin_check        VARCHAR2(20),
        freq_cap_check     VARCHAR2(20),
        control_group      VARCHAR2(20),
        review_queue_id    NUMBER,
        status             VARCHAR2(20),
        error_text         VARCHAR2(4000),
        retention_until    DATE
    )""",
    """CREATE TABLE offer_decision_log (
        decision_id        NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        decided_at         TIMESTAMP DEFAULT SYSTIMESTAMP NOT NULL,
        customer_id        NUMBER,
        use_case           VARCHAR2(30),
        trigger_event_id   NUMBER,
        candidate_offers   VARCHAR2(400),
        chosen_offer_id    NUMBER,
        decision           VARCHAR2(30),
        decision_reason    VARCHAR2(400),
        channel            VARCHAR2(20),
        channel_of_record  VARCHAR2(20),
        control_group      VARCHAR2(20),
        ai_call_id         NUMBER,
        trace_id           VARCHAR2(64),
        retention_until    DATE
    )""",
    """CREATE TABLE offer_suppression (
        customer_id NUMBER,
        channel     VARCHAR2(20),
        reason      VARCHAR2(200),
        created_at  TIMESTAMP DEFAULT SYSTIMESTAMP,
        PRIMARY KEY (customer_id, channel)
    )""",
    """CREATE TABLE do_not_contact (
        customer_id NUMBER PRIMARY KEY,
        reason      VARCHAR2(200),
        created_at  TIMESTAMP DEFAULT SYSTIMESTAMP
    )""",
    """CREATE TABLE marketing_policy (
        policy_id          NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        channel            VARCHAR2(20),
        freq_cap           NUMBER,
        freq_cap_window    INTERVAL DAY TO SECOND,
        quiet_hours_start  NUMBER,
        quiet_hours_end    NUMBER,
        effective_from     TIMESTAMP DEFAULT SYSTIMESTAMP
    )""",
    """CREATE TABLE approved_disclosures (
        disclosure_id   VARCHAR2(64) PRIMARY KEY,
        offer_id        NUMBER,
        effective_date  DATE,
        disclosure_text CLOB,
        created_by      VARCHAR2(64),
        approved_at     TIMESTAMP
    )""",
    """CREATE TABLE udaap_review_queue (
        review_id    NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        call_id      NUMBER,
        reason       VARCHAR2(40),
        state        VARCHAR2(20) DEFAULT 'PENDING',
        reviewer     VARCHAR2(64),
        reviewed_at  TIMESTAMP,
        notes        VARCHAR2(4000)
    )""",
    "CREATE INDEX ai_call_log_cust_ix ON ai_call_log(customer_id, created_at)",
    "CREATE INDEX ai_call_log_trace_ix ON ai_call_log(trace_id)",
    "CREATE INDEX odl_cust_ix ON offer_decision_log(customer_id, decided_at)",
    "CREATE INDEX odl_trace_ix ON offer_decision_log(trace_id)"
]

for stmt in governance_statements:
    try:
        run_sql(stmt, fetch=False)
    except Exception as e:
        # ORA-00955 = name is already used by an existing object
        if "ORA-00955" not in str(e):
            print(f"Statement skipped: {e}")
print("Governance tables/indexes created or already exist.")


In [ ]:
nudge_agent_statements = [
    """DECLARE
  user_exists NUMBER;
BEGIN
  SELECT COUNT(*) INTO user_exists FROM dba_users WHERE username = 'NUDGE_AGENT';
  IF user_exists = 0 THEN
    EXECUTE IMMEDIATE 'CREATE USER nudge_agent IDENTIFIED BY "ReplaceWithStrongSecret#123"';
  END IF;
END;""",
    "ALTER USER nudge_agent DEFAULT TABLESPACE users QUOTA 0 ON users",
    "ALTER USER nudge_agent TEMPORARY TABLESPACE temp"
]

for stmt in nudge_agent_statements:
    try:
        run_sql(stmt, fetch=False)
    except Exception as e:
        print(f"NUDGE_AGENT setup step skipped: {e}")
print("NUDGE_AGENT user configured (or setup requires ADMIN/DBA privileges).")
print("Next: grant EXECUTE only on named-tool wrappers; enable Unified Audit on NUDGE_AGENT via your bank's SIEM pipeline.")


# Select AI: Governed Generation, Not Open-Ended Text

`DBMS_CLOUD_AI` lets the database call an LLM for natural-language SQL generation and text generation. In this demo it is used for UC3 to craft a declined-transaction message. In production, the generation step is the **last and most controlled** step:

1. The offer has already been chosen.
2. Eligibility, suppression, opt-in, and frequency-cap checks have already passed.
3. The channel-of-record decision has already been made.
4. The LLM's job is to phrase a pre-approved decision — **not to make one.**

## The wrapper-package pattern

Direct calls to `DBMS_CLOUD_AI.GENERATE` from application code are a control failure. Every call should go through a wrapper package that:

- Re-runs opt-in / suppression / frequency-cap checks (defense in depth).
- Enforces the channel-of-record split (marketing vs. servicing).
- Substitutes approved Reg Z / Reg DD disclosure language into the output; the LLM never authors APR, APY, or fee text.
- Writes an `AI_CALL_LOG` row before and after the call with trace ID, prompt hash, output hash, and model version.
- Routes output to a UDAAP review queue when required (new template, new offer, new model, or sampled).
- Returns a deterministic fallback string on error.

## Key controls

- `object_list` in the profile is the allow-list of tables the LLM can see for NL→SQL grounding. Keep it minimal and remove columns like `CUSTOMER.full_name` from the surface.
- The model provider call leaves the bank; confirm the data-handling contract covers NPI and the region is approved.
- UC3 (declined transaction) is **servicing** under Reg E, not marketing. Marketing opt-out does not block it, but channel consent still applies.
- `AI_CALL_LOG` is a record-of-record table: it must include the literal output text, not just a hash, because regulators have asked for it.

## `AI_CALL_LOG` schema

```sql
CREATE TABLE ai_call_log (
  call_id            NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  created_at         TIMESTAMP DEFAULT SYSTIMESTAMP NOT NULL,
  customer_id        NUMBER,
  use_case           VARCHAR2(30),
  offer_id           NUMBER,
  channel            VARCHAR2(20),
  channel_of_record  VARCHAR2(20),
  profile_name       VARCHAR2(128),
  model_name         VARCHAR2(256),
  model_version      VARCHAR2(64),
  trace_id           VARCHAR2(64),
  span_id            VARCHAR2(32),
  prompt_template_id VARCHAR2(64),
  prompt_hash        VARCHAR2(128),
  prompt_tokens      NUMBER,
  output_tokens      NUMBER,
  output_hash        VARCHAR2(128),
  output_text        CLOB,
  disclosure_id      VARCHAR2(64),
  suppression_check  VARCHAR2(20),
  optin_check        VARCHAR2(20),
  freq_cap_check     VARCHAR2(20),
  control_group      VARCHAR2(20),
  review_queue_id    NUMBER,
  status             VARCHAR2(20),
  error_text         VARCHAR2(4000),
  retention_until    DATE
);
```

All three `*_check` columns must be `PASS` for `status = OK`. The wrapper enforces this; Module 6 integrity checks verify it.

## UDAAP review queue and approved disclosures

A percentage of generated nudges — and 100% of nudges using a new template, new offer, or new model — is routed to a human `UDAAP_REVIEW_QUEUE` before or after delivery. The wrapper inserts into this queue when policy says so.

The LLM must **never** paraphrase APR, APY, fees, or rate terms. The wrapper renders the prompt from an approved template with a placeholder like `{{disclosure_block}}`, calls `GENERATE`, then **replaces** the placeholder with pre-approved text from an `APPROVED_DISCLOSURES` table keyed by `offer_id` and effective date. If the placeholder is missing or the LLM output contains a numeric `%` outside the placeholder, the wrapper rejects the output.

## Use-case-specific governance

| UC | Trigger | Channel of record | Most-relevant rules |
|---|---|---|---|
| UC1 — credit-card page view | `/products/cash-plus-visa` | Marketing | UDAAP, Reg Z if APR mentioned, TCPA/CAN-SPAM if pushed off-page |
| UC2 — abandoned application | `status='STARTED'` older than 1 hour | Marketing | UDAAP, Reg Z, ECOA (no protected-class attribute to choose who to nudge), CAN-SPAM/TCPA |
| UC3 — declined transaction | `txn.status='DECLINED'` | **Servicing (Reg E)** | Reg E error/dispute, BSA/AML (no SAR leakage), GLBA. **Not** a marketing message |

## SLOs (defensible starting numbers — calibrate to your bank)

| SLO | Target | Why |
|---|---|---|
| P95 end-to-end nudge latency | < 1200 ms | UC1/UC2 must complete inside the customer's session window |
| P99 DB retrieval latency | < 400 ms | Leaves budget for the LLM call |
| UC3 P95 latency | < 800 ms | Servicing message expected near-real-time |
| Availability (decision pipeline) | ≥ 99.9% | Below this, marketing campaigns leak conversions |
| Generated-text fallback rate | ≤ 0.5% | Higher means the LLM provider is degrading or templates are broken |
| Suppression-bypass incidents | 0 | Any non-zero count is a Sev-1 — UDAAP / TCPA exposure |

In [ ]:
counts = run_sql("""
SELECT (SELECT COUNT(*) FROM customer) AS customer_count,
       (SELECT COUNT(*) FROM txn) AS txn_count,
       (SELECT COUNT(*) FROM application) AS app_count,
       (SELECT COUNT(*) FROM conversation_chunk) AS chunk_count
FROM dual
""")
print("Row counts:")
print(counts)

vector_sanity = run_sql("""
SELECT chunk_text
FROM conversation_chunk
ORDER BY VECTOR_DISTANCE(
  embedding,
  VECTOR_EMBEDDING(MINILM_EMB USING 'card comparison request' AS DATA),
  COSINE)
FETCH FIRST 3 ROWS ONLY
""")
print("Vector sanity:")
print(vector_sanity)

# Use Case 1: Card Page View Nudge

When a customer views a card product, the system finds peer products through graph traversal and ranks relevant conversation snippets with vector similarity. The query below binds a sample `customer_id`.

In [ ]:
uc1_sql = """
WITH last_view AS (
  SELECT product_id
  FROM page_event
  WHERE customer_id = :cid
  ORDER BY event_ts DESC
  FETCH FIRST 1 ROW ONLY
),
peer_products AS (
  SELECT *
  FROM GRAPH_TABLE(
    banking_graph
    MATCH (c1 IS customer)-[:viewed]->(p IS product)<-[:viewed]-(c2 IS customer)-[:viewed]->(p2 IS product)
    WHERE c1.customer_id = :cid
      AND p.product_id = (SELECT product_id FROM last_view)
    COLUMNS (
      p2.product_id AS peer_product_id,
      p2.name AS peer_product
    )
  )
)
SELECT p.peer_product,
       cc.chunk_text,
       VECTOR_DISTANCE(
         cc.embedding,
         VECTOR_EMBEDDING(MINILM_EMB USING 'credit card comparison help' AS DATA),
         COSINE
       ) AS distance
FROM conversation_chunk cc
CROSS JOIN peer_products p
ORDER BY distance
FETCH FIRST 5 ROWS ONLY
"""

uc1_result = run_sql(uc1_sql, {"cid": 1001})
print(uc1_result)

# Use Case 2: Abandoned Application Recovery

Applications in `STARTED` status older than one hour are matched to similar past conversation snippets. The result provides the context needed to craft a recovery nudge.

In [ ]:
uc2_sql = """
WITH abandoned AS (
  SELECT a.app_id, a.customer_id, a.product_id, a.updated_at, a.fields_json
  FROM application a
  WHERE a.status = 'STARTED'
    AND a.updated_at < SYSTIMESTAMP - INTERVAL '1' HOUR
)
SELECT ab.app_id, ab.customer_id, p.name AS product_name, cc.chunk_text,
       VECTOR_DISTANCE(
         cc.embedding,
         VECTOR_EMBEDDING(MINILM_EMB USING 'application abandoned income verification step' AS DATA),
         COSINE
       ) AS distance
FROM abandoned ab
JOIN product p ON p.product_id = ab.product_id
CROSS JOIN conversation_chunk cc
ORDER BY distance
FETCH FIRST 10 ROWS ONLY
"""

print(run_sql(uc2_sql))

# Use Case 3: Declined Transaction Explanation

A declined transaction triggers Select AI to generate an explainable, policy-safe nudge. The cell is wrapped in try/except so the notebook continues if Select AI is not configured.

In [ ]:
try:
    txn_context = run_sql("""
    SELECT t.txn_id, t.amount, t.status, t.decline_reason, c.customer_id, c.segment
    FROM txn t
    JOIN account a ON a.account_id = t.account_id
    JOIN customer c ON c.customer_id = a.customer_id
    WHERE t.status = 'DECLINED'
    FETCH FIRST 1 ROW ONLY
    """).iloc[0]

    prompt = (
        f"Customer {txn_context['CUSTOMER_ID']} ({txn_context['SEGMENT']} segment) "
        f"just had a declined transaction of ${txn_context['AMOUNT']} "
        f"with reason '{txn_context['DECLINE_REASON']}'. "
        "Craft a one-sentence proactive, policy-safe nudge explaining the decline and the next step."
    )

    uc3_result = run_sql("SELECT DBMS_CLOUD_AI.GENERATE(prompt => :p, action => 'chat') AS nudge FROM dual", {"p": prompt})
    print(uc3_result)
except Exception as e:
    print(f"UC3 skipped: {e}")
    print("This use case requires Select AI profile NUDGE_BOT and OCI GenAI credentials.")

# APEX Integration

The APEX application export lives at `26ai-banking-demo/apex/nudge_chat_app.sql`. Import it into APEX to get the chat UI. The same backend logic is available from the PL/SQL package below, which an APEX page process can call as `nudge_chat_api.get_nudge('UC1', 1001)`.

# MCP Is a Policy Enforcement Point, Not Just an API

MCP (Model Context Protocol) lets an LLM agent call **explicitly named tools** with typed parameters. For a bank, that turns "an LLM with a database password" into "an LLM with a fixed catalog of audited, least-privilege actions."

## A defensible initial tool catalog

| Tool | Purpose | Channel of record |
|---|---|---|
| `peer_products(cid, limit)` | UC1 candidate generation via `BANKING_GRAPH` | Marketing |
| `recent_declines(cid, lookback_hours)` | UC3 trigger lookup | Servicing |
| `similar_chunks(query_text, top_k, customer_id)` | Vector retrieval; requires `customer_id` so opt-in/suppression are enforced | Inherits from caller |
| `is_eligible(cid, offer_id)` | Deterministic eligibility check | n/a |
| `is_suppressed(cid, channel, use_case)` | Suppression + opt-out + frequency cap | n/a |
| `generate_nudge(...)` | Calls the wrapper package; re-runs eligibility and suppression | Per use case |
| `record_decision(...)` | Writes to `OFFER_DECISION_LOG` | n/a |

## The catalog must never include

- Tools that return raw `transcript`, `full_name`, account number, PAN, SSN, or DOB.
- Tools that bypass `is_suppressed` for marketing channels.
- Tools that let the credit-decision path deny an application based on graph or vector output.
- Tools with `EXECUTE IMMEDIATE` or arbitrary SQL passthrough.

## Least-privilege agent user

The DB user the SQLcl MCP server connects as should be:

- A dedicated `NUDGE_AGENT` user, never `ADMIN`.
- No system privileges. Specifically: no `CREATE TABLE`, no `CREATE PROCEDURE`, no `EXECUTE ANY PROCEDURE`, no `SELECT ANY TABLE`, no `ALTER SESSION` other than what `connection-init-sql` requires.
- Object grants only on the named-tool wrappers — not on underlying tables, not on `DBMS_CLOUD_AI` directly.
- Resource Manager consumer-group cap so a runaway agent cannot starve OLTP.
- Connection from the MCP server only (DB-side access control by client IP / ACL / mTLS).
- Full unified audit on the `NUDGE_AGENT` schema; ship audit to SIEM.

## Suppression and opt-out — wrappers, not advice

The agent has no ability to bypass suppression. A tool like `generate_nudge` calls `pkg_nudge_policy.is_suppressed(cid, channel, use_case)`; if it returns 'Y', the tool returns a deterministic "suppressed" response and writes a `SUPPRESSED` row to the decision log. It never calls the LLM.

Rules baked into the function:

- If `use_case = 'SERVICING'` (UC3), marketing opt-out is ignored, but channel-specific consent is still enforced.
- If `customer.personalization_opt_in = 'N'` and use case is marketing → suppressed.
- If a row exists in `offer_suppression(customer_id, channel)` → suppressed.
- If a row exists in `do_not_contact(customer_id)` → suppressed (TCPA do-not-call, CAN-SPAM unsubscribe, account-level holds).
- Frequency-cap: count `ai_call_log` rows for `(customer_id, channel)` in the rolling window from `marketing_policy.freq_cap`.
- Quiet-hours: check `customer.timezone` against `marketing_policy.quiet_hours_start/end` for time-of-day-sensitive channels.

## Per-tool logging

Every tool invocation writes caller identity, tool name + version, input parameters (PII redacted), W3C traceparent, result summary (row counts, decision codes — not raw NPI), elapsed time, and error class. This log feeds the same SIEM stream as `AI_CALL_LOG`; a regulator data request joins them on `customer_id` + `trace_id`.

In [ ]:
apex_pkg = """
CREATE OR REPLACE PACKAGE nudge_chat_api AS
  FUNCTION get_nudge(p_use_case IN VARCHAR2, p_customer_id IN NUMBER) RETURN CLOB;
END nudge_chat_api;
/

CREATE OR REPLACE PACKAGE BODY nudge_chat_api AS
  FUNCTION get_nudge(p_use_case IN VARCHAR2, p_customer_id IN NUMBER) RETURN CLOB IS
    l_out CLOB;
  BEGIN
    IF p_use_case = 'UC1' THEN
      SELECT TO_CLOB('I see you viewed a card product recently. Want a quick comparison?')
      INTO l_out FROM dual;
    ELSIF p_use_case = 'UC2' THEN
      SELECT TO_CLOB('Looks like your application is still in progress. Need help to finish it?')
      INTO l_out FROM dual;
    ELSIF p_use_case = 'UC3' THEN
      SELECT DBMS_CLOUD_AI.GENERATE(
               prompt => 'Customer ' || p_customer_id || ' just had a declined transaction. Craft a one-sentence proactive nudge.',
               action => 'chat'
             )
      INTO l_out FROM dual;
    ELSE
      l_out := TO_CLOB('Unsupported use case. Use UC1, UC2, or UC3.');
    END IF;
    RETURN l_out;
  END get_nudge;
END nudge_chat_api;
/
"""

try:
    run_sql(apex_pkg, fetch=False)
    print("Package nudge_chat_api created.")
    print(run_sql("SELECT nudge_chat_api.get_nudge('UC1', 1001) AS nudge FROM dual"))
except Exception as e:
    print(f"APEX package creation skipped: {e}")

# MCP Integration

MCP (Model Context Protocol) lets an LLM call **explicitly defined tools** through SQLcl instead of writing arbitrary SQL. For a bank, that turns the integration from "an LLM with a database password" into "an LLM with a fixed catalog of named, typed, audited, least-privilege actions."

Install SQLcl 24+ and start the server:

```bash
sql -mcp \
  'user/"password"@NudgeDB_HIGH' \
  -proxy "http://proxy.example.com:80"
```

Example `claude_desktop_config.json` snippet:

```json
{
  "mcpServers": {
    "oracle-adb-nudges": {
      "command": "sql",
      "args": ["-mcp"],
      "env": {
        "TNS_ADMIN": "/absolute/path/to/wallet_NudgeDB",
        "JAVA_HOME": "/absolute/path/to/jdk-17"
      }
    }
  }
}
```

## Named-tool catalog (initial)

| Tool | Purpose | UC | Channel |
|---|---|---|---|
| `peer_products(cid, limit)` | Peer-viewed products via `BANKING_GRAPH` | UC1 | Marketing |
| `recent_card_view(cid)` | Last card `page_event` | UC1 | Marketing |
| `abandoned_apps(lookback_hours)` | UC2 trigger sweep | UC2 | Marketing |
| `app_context(app_id)` | Grounding minus PII | UC2 | Marketing |
| `recent_declines(cid, lookback_hours)` | UC3 trigger lookup | UC3 | Servicing |
| `decline_explanation(txn_id)` | Deterministic decline reason → approved language | UC3 | Servicing |
| `similar_chunks(query_text, top_k, customer_id)` | Vector retrieval with consent/suppression gates | All | Inherits caller |
| `is_eligible(cid, offer_id)` | Deterministic eligibility check | All | n/a |
| `is_suppressed(cid, channel)` | Suppression + opt-out + frequency-cap | All | n/a |
| `generate_nudge(cid, offer_id, use_case, channel)` | Calls `PKG_NUDGE_AI.GENERATE` (defense-in-depth re-checks) | All | Per use_case |
| `record_decision(...)` | Writes `OFFER_DECISION_LOG` | All | n/a |

Example agent prompts:
- "Find recent declined transactions and explain likely reasons for customer 1001."
- "Use graph traversal to list products peers viewed after Cash+ Visa."
- "Retrieve similar abandoned-application chats and draft a one-line nudge."

# Spring Integration Reference

The Spring/Java example files in `26ai-banking-demo/examples/spring/` show production wiring:

- `application.yml` configures the datasource, wallet path, and sets `DBMS_CLOUD_AI.SET_PROFILE('NUDGE_BOT')` on connection.
- `NudgeRepository.java` runs the UC1/UC2/UC3 queries.
- `NudgeService.java` adds OpenTelemetry spans around each use case.
- `OtelDataSourceConfig.java` instruments the datasource.

## Production checklist for Spring / service tier

- The service account used by the app tier has exactly the grants in the `nudge_app_grants.sql` section — no more, no less.
- `DBMS_CLOUD_AI.SET_PROFILE` is executed once per session, typically via `connection-init-sql` in the pool.
- The wallet directory is mounted read-only; credentials are never checked into source control.
- Every public method in `NudgeService` is annotated with `@Timed` and `@Counted` and emits OpenTelemetry spans.
- Circuit-breaker configured on Select AI calls with a deterministic fallback to pre-approved templates.
- PII never serialized in span attributes or logs; include `customer_id` only when required by a trace header.

These files are kept in the repo for reference and are not executed inside this notebook.

# Capacity Planning in Detail

Before rollout, estimate the vector storage footprint with real dimensions and index parameters.

## Baseline formulas

$$bytes \approx dims \times 4$$

$$raw\_total \approx rows \times dims \times 4$$

Then add operational overhead:

- Data segment: 1.2× to 1.5× of raw_total
- Vector index: 0.5× to 1.5× of raw_total (depends on IVF vs. HNSW and parameters)
- Growth and maintenance headroom: +25% to +40%

## Worked example: 2,000,000 offers × 384 dims

- Raw embeddings: 2,000,000 × 384 × 4 = 3.07 GB
- Data segment (1.3×): ~4 GB
- Vector index (1.0×): ~3 GB
- Subtotal: ~7 GB
- With 30% headroom: **provision ~9.1 GB**

For conversation chunks at 10,000 rows × 384 dims, the footprint is tiny (~15 MB raw), which is why the demo can use IVF without thinking about memory. At bank scale, measure before choosing HNSW vs. IVF.

## Measurement loop

Run a pilot load with production-like dimensions, capture `USER_SEGMENTS` before and after embedding + index build, and re-measure after one re-embedding cycle. Use that delta as the canonical number for capacity planning.

## Capacity per use-case traffic mix

- **UC1** dominates steady-state traffic (page views).
- **UC2** is bursty after the abandoned-application batch sweep.
- **UC3** follows real-time decline-rate spikes.
- Size `top_K`, connection pool, and MCP concurrency separately for each mix; do not size for average traffic if burst headroom is required.

# Deployment topology and rollback

A bank-grade deployment separates environments at the database layer, not only the app layer:

| Environment | Purpose | Data |
|---|---|---|
| Development | Feature work, notebook replay | Synthetic or masked sample |
| Model lab | Embedding model evaluation, Select AI prompt testing | Masked production sample under DUA |
| QA / UAT | End-to-end offer flow, UDAAP copy review, compliance sign-off | Masked, no production credentials |
| Production | Live customers | Live, full audit, legal-hold enabled |

## Rollback plan

Before enabling a new model, prompt template, or policy rule, prepare a rollback:

1. Capture a snapshot of the prior vector index, ONNX model, and Select AI profile name.
2. Keep the prior version as a named asset (`conversation_chunk_v1`, `minilm_v1`, `NUDGE_BOT_V1`).
3. Gate rollout behind a feature flag keyed to `offer_config` so it can be disabled in one update.
4. Automate the rollback decision from SLO burn-rate: if P95 latency > SLO for 10 minutes or fallback rate > 1%, switch to the previous version.

# Launch-readiness checklist

## Architecture
- [ ] Architecture document approved by Architecture Review.
- [ ] Data-flow diagram reviewed by InfoSec; PII surfaces identified.
- [ ] Data-flow inventory updated (Modules 1, 3, 4) — NPI paths and third-party egress identified.
- [ ] Data residency confirmed for OCI GenAI region.
- [ ] Vector/graph access paths are read-only for the app account; write is limited to batch jobs.
- [ ] Network path from app to database uses mTLS or TLS 1.3; no plaintext DB ports.
- [ ] Feature flags and kill-switches exist for model, prompt, policy, and channel.

## Models and embeddings (SR 11-7)
- [ ] `MINILM_EMB` in model inventory with owner, version, validation report, intended-use statement, and monitoring plan.
- [ ] `cohere.command-r-plus` (or chosen LLM) in model inventory + vendor data-handling terms reviewed.
- [ ] Challenger / fallback strategy documented for both models.
- [ ] Embedding model is approved by model risk; version, checksum, and license recorded.
- [ ] Recall canary passes with production-like queries (see Module 6).
- [ ] HNSW/IVF parameters chosen from measured build time and recall, not defaults.

## Privacy and security
- [ ] GLBA NPI mapping covers `CONVERSATION`, `CONVERSATION_CHUNK`, `AI_CALL_LOG.output_text`.
- [ ] GDPR/CCPA erasure path deletes from all three.
- [ ] All source text reviewed for NPI; redaction/VPD applied where needed.
- [ ] `AI_CALL_LOG` and `OFFER_DECISION_LOG` capture literal outputs and decisions.
- [ ] `NUDGE_AGENT` / app account privileges are least-privilege and audited.
- [ ] Encryption at rest (TDE) + in transit (TLS) verified.
- [ ] Audit ingestion to SIEM verified.
- [ ] Wallet/TLS credentials rotated and stored in a secrets manager; not in GitHub.

## Compliance
- [ ] UDAAP, Reg B/ECOA, Reg Z/DD, Reg E, FCRA, TCPA/CAN-SPAM, GLBA, GDPR/CCPA reviews complete.
- [ ] UDAAP review queue policy approved (sampling rate, gating rules).
- [ ] Reg Z / Reg DD disclosure templates approved and loaded into `APPROVED_DISCLOSURES`.
- [ ] Reg E classification of UC3 confirmed in writing.
- [ ] ECOA / Reg B sign-off on UC1 graph features and UC2 eligibility.
- [ ] FCRA: no LLM-authored adverse-action reasons. Confirmed.
- [ ] CAN-SPAM / TCPA: opt-in evidence available per channel; quiet-hours enforced.
- [ ] Marketing opt-out and suppression lists loaded; no nudge route bypasses them.
- [ ] Disclosure language for APR, APY, fees is static and approved; LLM only substitutes variables.
- [ ] Holdout group is implemented and attribution window is documented.
- [ ] Fair-lending disparate-impact monitoring configured with demographic parity thresholds.

## Operations
- [ ] On-call runbook (Module 6) validated.
- [ ] SLOs and alerts wired.
- [ ] Cost dashboard and budgets configured.
- [ ] Golden signals dashboard: latency (P50/P95/P99), throughput, error rate, fallback rate, cost per nudge, suppression-bypass count.
- [ ] Paging alerts for suppression-bypass, fallback rate spike, latency SLO miss, and cost anomaly.
- [ ] Runbooks for incident classes: model degradation, embedding drift, Select AI failure, suppression system outage, audit data request.
- [ ] Records-management retention configured on `AI_CALL_LOG` and `OFFER_DECISION_LOG`.
- [ ] Retention and legal-hold policy implemented; `retention_until` column populated.

## Decision quality
- [ ] Holdout / control group enabled per offer.
- [ ] Attribution job scheduled.
- [ ] Disparate-impact monitoring scheduled.

If any box is unchecked, you don't ship.

Use the SQL below to capture before/after segment snapshots when scaling.

In [ ]:
space = run_sql("""
SELECT segment_name, segment_type, bytes/1024/1024 AS mb
FROM user_segments
WHERE segment_name IN ('CONVERSATION_CHUNK', 'CONV_CHUNK_IDX')
ORDER BY segment_name
""")
print(space)

# Operations, Observability, and Audit (Module 6)

In a regulated bank, the ops dashboard is also evidence. A regulator will ask:

- "Show me, for this customer, every nudge they were shown and every nudge they were *not* shown, with reasons."
- "Show me your model monitoring for the last 12 months."
- "Show me the time-to-resolution distribution of your UDAAP review queue."
- "Show me proof that opt-out was honored on date X."
- "Show me the disparate-impact analysis on offer presentation by protected-class proxy."

## Golden signals + banking-specific signals

| Signal | Instrument | Action threshold |
|---|---|---|
| End-to-end latency (P50/P95/P99) | OpenTelemetry spans / Micrometer | Page on P95 > SLO for 5 min |
| DB retrieval latency | SQL monitor / AWR | Page on P99 > 400 ms |
| Throughput | Micrometer counters | Track per UC/channel |
| Error rate | Spans + `OFFER_DECISION_LOG.decision='ERROR'` | Page on > 0.1% |
| Fallback rate | `AI_CALL_LOG.fallback_flag` | Page on > 0.5% |
| Cost per nudge | Cloud cost labels / LLM token metrics | Page on > 2× baseline |
| Suppression-bypass count | `OFFER_DECISION_LOG` + audit | **Any non-zero = Sev-1** |
| Disclosure-substitution failure | `AI_CALL_LOG` missing placeholder | **Any non-zero = Sev-1** |
| UDAAP-review queue depth | Queue table | Page if oldest > SLA |
| Graph query timeout | `GRAPH_TABLE` timeout telemetry | Page on > 0.1% |

## Vector plan inspection

After running a top-K query, confirm it used the vector index:

```sql
SELECT * FROM TABLE(DBMS_XPLAN.DISPLAY_CURSOR(format => 'ALLSTATS LAST'));
```

Look for `VECTOR INDEX` access. If you see a full scan, the optimizer has rejected the index — latency will explode at scale. Common causes: wrong distance function, type mismatch, or statistics stale. Capture a SQL Plan Baseline for the canonical query:

```sql
BEGIN
  DBMS_SPM.LOAD_PLANS_FROM_CURSOR_CACHE(sql_id => '<canonical_sql_id>');
END;
/
```

## AWR / ASH focus areas

- `DB time` by SQL ID for the UC1/UC2/UC3 canonical queries.
- Wait events: `cell single block physical read` (storage), `vector index read` (normal), `resmgr:cpu quantum` (Resource Manager throttling).
- `PGA` and `TEMP` usage during HNSW build or large-batch embeddings.
- `ASH` samples around Select AI calls to spot network / provider latency.

## Recall canary (SR 11-7 ongoing monitoring)

Maintain a fixed list of `(query_text, expected_chunk_id)` canary pairs. Per release, measure recall@K for the ANN index against exact-distance ground truth. A drop beyond your policy threshold blocks release. This is how you satisfy SR 11-7's ongoing monitoring requirement for the embedding model.

## Resource Manager containment

Map the app account to a consumer group with CPU cap so runaway MCP or batch embeddings cannot starve OLTP:

```sql
BEGIN
  DBMS_RESOURCE_MANAGER.CREATE_CONSUMER_GROUP('NUDGE_BATCH_CG', 'Nudge batch embedding group');
  DBMS_RESOURCE_MANAGER.SET_CONSUMER_GROUP_MAPPING(
    attribute => 'CLIENT_OS_USER', value => 'nudge_batch', consumer_group => 'NUDGE_BATCH_CG'
  );
END;
/
```

## OpenTelemetry and Micrometer integration

The Spring examples add `@Timed` and `@Counted`. In Python, use `opentelemetry-api/sdk` and `prometheus-client`:

- Span per use case with attributes: `use_case`, `channel`, `decision` (no PII).
- Counter `nudge_decisions_total{decision,use_case,channel}`.
- Histogram `nudge_latency_seconds` buckets calibrated to SLO.
- Export to OTLP collector; route metrics to Prometheus/Grafana and traces to Jaeger/Tempo.

## Disparate-impact monitoring (Reg B / ECOA + Fair Lending)

This is mandatory for credit-product offers. Schedule a daily/weekly job that samples `OFFER_DECISION_LOG` joined to `CUSTOMER`, computes presentation and suppression rates by `segment` (and any other monitored attribute approved by Compliance) for each credit-family offer, and runs a statistical test (e.g., proportions test). Output goes to a controlled compliance report, not a casual dashboard tile.

## Cost controls

- Set cloud provider spend alerts on the Autonomous Database and LLM provider at 50%, 80%, and 100% of budget; page at 80%.
- Cap concurrency in the MCP server and connection pool.
- Cache top-K results for high-cardinality queries; only re-embed changed conversations.
- Reconcile `AI_CALL_LOG.prompt_tokens + output_tokens` by day to OCI billing exports. Variance above a threshold is investigated (a SOX-relevant control if attribution feeds revenue reporting).
- Use ADB Free Tier for dev/test; tag all production resources with `cost-center` and `env`.

## Retention, legal hold, and erasure

- `AI_CALL_LOG.retention_until` and `OFFER_DECISION_LOG.retention_until` are computed at insert time (typical: 7 years for marketing, longer for credit decisions).
- A purge job checks a `LEGAL_HOLD` table first; records on hold are exempt regardless of `retention_until`.
- GDPR/CCPA erasure deletes before retention expiry only when no legal hold applies, and must cascade to `CONVERSATION` and `CONVERSATION_CHUNK` embeddings.

## Alert set and incident playbooks

| Alert | Severity | Initial response |
|---|---|---|
| Suppression bypass detected | 1 | Disable offending tool/path; page Compliance + Legal |
| Disclosure substitution failure | 1 | Stop Select AI; route to fallback; page Compliance |
| P95 latency > SLO 10 min | 2 | Roll back to previous model/index; investigate plan |
| Fallback rate > 1% | 2 | Check LLM provider status; verify template health |
| Vector plan full scan | 2 | Gather SQL monitor; force baseline or refresh stats |
| Cost > 2× baseline | 3 | Review cache hit rate and concurrency caps |
| UDAAP queue > SLA | 3 | Reassign reviewers; pause new templates |
| ADB storage > 80% | 2 | Scale storage; review retention/purge |

## Daily operations checklist

- [ ] Review suppression-bypass and disclosure-failure dashboards — both must be zero.
- [ ] Check vector index status and plan stability for canonical queries.
- [ ] Validate last successful embedding refresh and CDC lag.
- [ ] Inspect UDAAP review queue depth and oldest item.
- [ ] Confirm legal-hold table is current before any purge job runs.
- [ ] Spot-check `OFFER_DECISION_LOG` negative decisions for sensible `decision_reason` values.

## Security controls

- **Row-level security and redaction**: apply VPD/RLS and redaction policies to source text and vector columns so embeddings do not leak privileged data.
- **Audit retrieval inputs, candidates, and decision payloads**: log the trigger context, the retrieved vector/graph candidates, and the generated nudge.
- **Deterministic fallbacks**: when vector or graph paths degrade, fall back to rule-based offers or cached top-performing nudges.
- **Failure domains and graceful degradation**:
  - CDC lag → use synchronous triggers or bounded staleness checks.
  - Embedding/index lag → refresh job with monitoring on `conversation_chunk` lag.
  - Graph timeout → cap hops and timeout; fall back to direct product rules.
  - Channel delivery failure → queue nudges and retry with exponential backoff.

## Verify-yourself checks

Run these before claiming the demo is ready:

1. Connect with the **app account**, not ADMIN; confirm `SELECT` on only the intended views/packages.
2. Query `OFFER_DECISION_LOG` for a suppressed customer and confirm `decision='SUPPRESSED'` with a reason.
3. Verify `AI_CALL_LOG` contains the literal generated text, prompt hash, model version, and trace ID.
4. Explain a nudge end-to-end: trigger → candidates → decision → generation → dispatch → archive.
5. Show the holdout group has no `ai_call_id` and no delivery event.
6. Demonstrate the fallback path works when Select AI is unavailable.

# Demo Walkthrough and Final Review

## Suggested demo narrative

Use this flow when presenting the notebook to stakeholders:

1. **Regulatory framing** — every nudge is a regulated communication; controls are not optional.
2. **Architecture** — show the mermaid diagram; point out retrieval, decision, generation, and archive layers.
3. **Connect safely** — explain Codespaces secrets / wallet / TLS and why credentials are outside the notebook.
4. **Load data** — run the schema, staging, transform, embedding, and graph cells.
5. **Run UC1** — card page view → peer products via graph → ranked conversation snippets via vector search.
6. **Run UC2** — abandoned application → vector search for similar past chats.
7. **Run UC3** — declined transaction → Select AI generates a policy-safe explanation.
8. **Show audit** — `OFFER_DECISION_LOG` and `AI_CALL_LOG` connect the nudge back to source rows.
9. **Show controls** — suppression, opt-out, frequency cap, disclosure substitution, holdout.
10. **Show operations** — golden signals, plan inspection, incident playbooks, daily checklist.

## Final verify-yourself checklist

Before the demo is "production storytelling ready":

- [ ] Notebook runs top-to-bottom without errors (or graceful skips for optional capabilities).
- [ ] All credentials come from environment variables; no secrets in cells or outputs.
- [ ] `AI_CALL_LOG` and `OFFER_DECISION_LOG` rows are produced and explainable.
- [ ] At least one suppressed/holdout customer can be demonstrated with a reason.
- [ ] The vector query plan shows `VECTOR INDEX` access.
- [ ] The rollback plan and SLO table can be discussed from the notebook text.
- [ ] The `.gitignore` blocks wallets, `.env`, and checkpoints.

In [ ]:
RUN_CLEANUP = False

cleanup_statements = [
    "DROP PROPERTY GRAPH banking_graph",
    "DROP INDEX conv_chunk_idx",
    "DROP TABLE conversation_chunk PURGE",
    "DROP TABLE conversation PURGE",
    "DROP TABLE page_event PURGE",
    "DROP TABLE application PURGE",
    "DROP TABLE txn PURGE",
    "DROP TABLE account PURGE",
    "DROP TABLE offer PURGE",
    "DROP TABLE product PURGE",
    "DROP TABLE customer PURGE",
    "DROP TABLE stg_paysim PURGE",
    "DROP TABLE stg_lending PURGE",
    "DROP TABLE stg_banking77 PURGE",
    "DROP TABLE stg_marketing PURGE",
    "BEGIN DBMS_VECTOR.DROP_ONNX_MODEL('MINILM_EMB'); EXCEPTION WHEN OTHERS THEN NULL; END;"
]

if RUN_CLEANUP:
    for stmt in cleanup_statements:
        try:
            run_sql(stmt, fetch=False)
        except Exception as e:
            print(f"Cleanup step skipped: {e}")
    print("Cleanup complete.")
else:
    print("Cleanup skipped. Set RUN_CLEANUP = True to drop demo objects.")

# Summary and Next Steps

This notebook demonstrated:

- A self-contained Oracle 26ai schema for proactive banking nudges.
- In-database ONNX embeddings and an AI Vector Search index.
- A SQL/PGQ property graph overlay on relational tables.
- Select AI configuration for natural-language nudge generation.
- APEX and MCP integration patterns.
- Three runnable use cases: card page view, abandoned application, and declined transaction.

## Suggested Next Steps

- Run the notebook end-to-end against an ADB 26ai Free Tier instance.
- Replace the synthetic data pipeline with Oracle GoldenGate CDC from a real banking schema.
- Add hybrid vector indexes as the conversation corpus grows.
- Tune `TARGET ACCURACY` and neighbor partitions for production latency targets.
- Complete the engineering review checklist before rollout.

# Where to Go Next

This notebook is the technical foundation. The next steps to make it production-defensible are:

1. **Run it against ADB 26ai Free Tier** with real credentials and the public datasets.
2. **Add production controls:** opt-in, suppression, frequency-cap, approved-disclosure substitution, and the `AI_CALL_LOG` / `OFFER_DECISION_LOG` tables.
3. **Harden the MCP catalog** with a dedicated `NUDGE_AGENT` role and tool wrappers that call `is_suppressed` before any generation.
4. **Instrument everything** with OpenTelemetry, Micrometer metrics, and SIEM audit ingestion.
5. **Establish model-risk governance** for `MINILM_EMB` and the LLM provider, with recall canaries and disparate-impact monitoring.
6. **Conduct the launch-readiness checklist** (architecture, models, privacy, compliance, operations, decision quality) before any customer-facing rollout.

If you can explain every nudge in one paragraph — who, why, with what data, generated by which model, reviewed by whom, and retained for how long — you are ready to ship.